# Lab 08-1 - Evaluating Classifier Performance

This lab covers various approaches to understanding the performance of classification models.

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# KNN will warn you about future updates, but we'll turn these off
import warnings
warnings.simplefilter(action = 'ignore', category = FutureWarning)  

The examples in this lab will use the Wisconsin breast cancer data introduced in a previous lab. These data were collected in a study that explored using machine learning to diagnose breast cancer tissue samples $y$ as either:

- cancerous (a label of `'M'` for malignant) or
- non-cancerous (a label of `'B'` for benign)

i.e. $y$ is binary categorical. The $x$ features in this data set describe the average characteristics (radius, symmetry, etc.) of the cells collected in each tissue sample.

In [ ]:
wbc = pd.read_csv(filepath_or_buffer = "../data/wisc_bc.csv")
wbc.head()

To begin, we'll perform an 80-20 training-testing split, separate the outcome variable, and when creating the feature variables drop the `ID` column as well since it is an identification variable not an actual variable of information:

In [ ]:
## Train-test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(wbc, test_size = 0.2, random_state = 7)

## Separate the outcome variables
train_y = train['Label']
test_y = test['Label']

## Separate the feature variables
train_X = train.drop(columns = ['ID','Label'])
test_X = test.drop(columns = ['ID','Label'])

## A) Confusion Matrices and Performance Scores

To generate a confusion matrix, we need a set of predicted labels to compare with the actual labels i.e. classes present in the outcome variable $y$. Because we'd like the confusion matrix we're analyzing to reflect the performance of our methods on new data, we should use out-of-sample predictions found using cross-validation.

We can do this quickly with the `cross_val_predict()` function. It accepts any estimator that has `.fit()` and `.predict()` methods, which means we can give it either a single model or a pipeline whose final step is a model.

Look at use of `cross_val_predict()` below. The goal is to get a predicted label `'M'` or `'B'` for *every* row of `train_X`, where each prediction comes from a model that never saw that row. `cross_val_predict()` does this in one step:

1. Splits the training data into 5 folds using the folds we created in `k_folds`
2. For each fold in turn:
    - Calls `.fit()` on `pipe` using the other 4 folds only. This fits `PowerTransformer()`, `StandardScaler()`, and `KNeighborsClassifier()`, in that order.
    - Calls `.predict()` on the held-out fold. The held-out fold is transformed and scaled using what was learned from the other 4 folds, and then classified as `'M'` or `'B'`.
3. Puts the 5 sets of predictions back together in the original row order.

The result is one array the same length as `train_y`. Note that 5 different models were fit along the way, but none of them is returned.

In [ ]:
## Function imports
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold

## Pipeline: normalize -> scale -> fit classifier
pipe = Pipeline(steps = [
    ('transformer', PowerTransformer(method = 'yeo-johnson')),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors = 8))
])

## Randomly assign rows to 5 folds (random_state sets the seed so results are reproducible)
## We do this here so that we can reuse the fold structure later
k_folds = KFold(n_splits = 5, shuffle = True, random_state = 76)

## Get out-of-sample predictions using 5-fold CV
train_y_pred = cross_val_predict(estimator = pipe, X = train_X, y = train_y, cv = k_folds)

Note how the outputs are predicted labels $\widehat{y}$: `'M'` (malignant, i.e. 1) or `'B'` (benign, i.e. 0).

In [ ]:
# Look at the first twenty values: they are M or B
train_y_pred[0:20]

Now let's compare the true `train_y` with the `train_y_pred` we just obtained via a confusion AKA contingency matrix. The simplest way to produce a confusion matrix in `sklearn` is the `confusion_matrix()` function:

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(
    y_true = train_y, 
    y_pred = train_y_pred, 
    labels = ['M','B']
)

The `labels` argument governs the order of the rows and columns in the confusion matrix. In this example we provided the list `labels = ['M','B']` because it's most natural to view malignant tumors `'M'` as the "positive" class. If `labels = ` is not specified, the matrix will be ordered using the values that appear at least once in provided values of `y_true` or `y_pred` in alpha-numeric order.

This output is not easy to read unfortunately. Rather, let's use `ConfusionMatrixDisplay()` which will create a graphical display of the confusion matrix:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(
    y_true = train_y,
    y_pred = train_y_pred, 
    labels = ['M','B']
)
plt.show()

Observe how:
- Of the 158 + 14 = 172 true `M` malignant cases, this model correctly classified 158/172 = 91.9%. This is called the *sensitivity* of our model. 
- Of the 10 + 273 = 283 true `B` benign cases, this model correctly classified 273/283 = 96.5%. This is called the *specificity* of our model. 

## B) Question 1


- **Part a)**: Modify the pipeline used in the above example to create a new pipeline `pipe_q1a` to include the following steps: 
    1. Standardization
    2. Dimension reduction using PCA with 7 retained components
    3. $k$-nearest neighbors with 15 neighbors and inverse-distance weighting
- **Part b)**: Use the pipeline from Part a) to
    1. Generate cross-validated predictions `train_y_pred_q1`. Re-use the `k_folds` assignment of each observation to folds from earlier. 
    1. Generate a visualization of the confusion matrix using `'M'` as the positive class
- **Part c)**: Compare the sensitivity achieved by the `pipe` procedure from earlier with the `pipe_q1a` from this question. Which is better? 
- **Part d)**: Compare the specificity achieved by the `pipe` procedure from earlier with the `pipe_q1a` from this question. Which is better? 
- **Part e)**: When it comes to identifying cancerous cells as in this example, what matters more: high sensitivity or high specificity? Why?

In [ ]:
# Code for Q1.a)



In [ ]:
# Code for Q1.b)



In [ ]:
# Code for Q1.c)



In [ ]:
# Code for Q1.d)



## C) Receiver Operating Characteristic (ROC) Curve Analysis

Recall in SDS 291 when you covered logistic regression where the outcome variable was the fitted log-odds:

$$
\log\left(\frac{\widehat{p}}{1-\widehat{p}}\right) = \widehat{\beta}_0 + \widehat{\beta}_1 x_1 + \widehat{\beta}_2 x_2 + \cdots + \widehat{\beta}_p x_p
$$

which you could then convert to obtain fitted/predicted probabilities $\widehat{p}$ that $y=1$ by using the inverse-logit function:

$$
\widehat{p} = \frac{1}{1 + e^{-(\widehat{\beta}_0 + \widehat{\beta}_1 x_1 + \cdots + \widehat{\beta}_p x_p)}}
$$

You could then obtain predictions $\widehat{y}$ = 1 or 0, i.e. malignant or benign, by using a probability threshold or cutoff such as 0.5:

$$
\widehat{y} =
\begin{cases}
1 & \text{if } \widehat{p} \geq 0.5 \\
0 & \text{otherwise}
\end{cases}
$$

ROC curve analysis provides a nuanced view of the performance of a classification model. The fundamental idea behind an ROC curve is to evaluate the trade-off between false positives and false negatives at **all possible cutoffs**, not just 0.5. 

Our first step is to obtain predicted probabilities $\widehat{p}$ instead of predicted class labels $\widehat{y}$. We can do this by setting the `method` argument in `cross_val_predict()` to `'predict_proba'`:

In [ ]:
train_y_pred_prob = cross_val_predict(
    estimator = pipe, 
    X = train_X, 
    y = train_y,
    # Re-use fold structure: 
    cv = k_folds, 
    # New: return predicted probabilities instead of labels
    method = 'predict_proba'
)

# Look at the first twenty rows: they are the fitted probabilities of
# obtaining B and M respectively
train_y_pred_prob[0:20,]

It's important to recognize that the above produced outcome is an array containing a column for each category of the outcome (arranged in alpha-numeric order). Because of this we'll use the second column (i.e. index position 1) when creating our ROC curve: 

In [ ]:
# ROC curve plotting:
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_curve

RocCurveDisplay.from_predictions(
    y_true = train_y, 
    y_pred = train_y_pred_prob[:,1], 
    pos_label = 'M'
)
plt.show()

Note that the values used to create these ROC curves can be accessed and explored numerically:

In [ ]:
fpr, tpr, thresh = roc_curve(
    y_true = train_y, 
    y_score = train_y_pred_prob[:,1], 
    pos_label = 'M'
)
print(pd.DataFrame({'t': thresh, 'TPR': tpr,'FPR': fpr}))

## D) Question 2


- **Part a)**: Re-using the folding structure `k_folds`, display the ROC curve of your pipeline `pipe_q1a` from Question 1
- **Part b)**: Report the highest true positive rate that can be achieved by `pipe_q1a` with a false positive rate of at most 5% i.e. 0.05. This is similar logic to when using a p-value cutoff of $\alpha=0.05$ to reject the null hypothesis from SDS 291.

In [ ]:
# Code for Q2.a)



In [ ]:
# Code for Q2.b)



## E) Performance Metrics and Tuning

A nice feature of pipelines and functions like `GridSearchCV()` is the compatibility with different model performance criteria. The code below demonstrates how to find the best modeling pipeline using the cross-validated area under the ROC curve (AUC) as the performance criterion, which is a different type of score than True Positive and False Positive Rates we saw earlier.

In [ ]:
## Simple pipeline
pipe_simple = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())
])

## Parameters to try via grid search. Observe we are searching over
## 2 x 2 x 2 = 8 possible combinations of parameters and trying to 
## identify the best.
from sklearn.preprocessing import RobustScaler
params = {
    'scaler': [StandardScaler(), RobustScaler()],
    'classifier__n_neighbors': [6, 10],
    'classifier__weights': ['uniform','distance']
}

## Conduct grid search
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
    estimator = pipe_simple, 
    param_grid = params, 
    cv = k_folds, 
    # Observe our performance metric is roc_auc
    scoring = 'roc_auc').fit(train_X, train_y)

## Print results for all 8 parameter combinations as a data frame. The
## estimated roc_auc is in the `mean_test_score` columns
pd.DataFrame(grid.cv_results_)

Now how do we access the results in `grid`? We can pick out the best parameter combination and best `roc_auc` score as follows:

In [ ]:
print(grid.best_params_)
print(grid.best_score_)

The best parameter combination is

1. Using the `RobustScaler()`
1. Using $k$-nearest neighbors with 10 neighbors
1. Using distance weighting of the points

yielding an AUC of 97.8%. But now how can we use this best combination? By accessing the `.best_estimator_`. That way you don't have to redefine your entire model.

In [ ]:
print(grid.best_estimator_)
print("\n")

# Simple example use: Use this estimator to make predictions on new data.
# Note best_pipe has already been refit on all of train_X by GridSearchCV
best_pipe = grid.best_estimator_
best_pipe.predict(test_X)[0:20]

## F) Question 3


- **Part a)**: Re-using your pipeline `pipe_q1a` from Question 1 and the folding structure `k_folds`, perform a cross-validated grid search `grid_q3` over a parameter grid `params_q3`, using ROC AUC as the performance criterion, to select the optimal combination of:
    1. The number of retained principal components: 5, 7, or no PCA (`'passthrough'`)
    2. The number of neighbors: 13, 15, or 17
    3. The weighting scheme: distance or uniform weighting
- **Part b)**: Report in a markdown cell what the optimal combination was and what its associated AUC is
- **Part c)**: Use the estimator associated with this optimal combination to draw the ROC curve for the best approach from Part a)

In [ ]:
# Code for Q3.a)



In [ ]:
# Code for Q3.c)

